# Pol-ISARSpaceTarget Non-YOLO OBB Detector Comparison

This notebook compares oriented bounding box detectors **apart from YOLO**.

Framework:

- MMRotate / OpenMMLab rotated object detection toolbox.

Methods:

1. Rotated RetinaNet
2. Rotated Faster R-CNN
3. RoI Transformer
4. Oriented R-CNN
5. R3Det, if the config is available in the installed MMRotate version

Dataset:

- Pol-ISARSpaceTarget-1.0
- Existing `labelTxt` annotations are already DOTA-style quadrilateral labels:
  `x1 y1 x2 y2 x3 y3 x4 y4 class difficult`

Input:

- `single_vv` amplitude image by default, because it was the strongest
  previous OBB input.

Reference YOLO result from earlier:

- YOLO11n-OBB + single_vv: mAP50 = 0.821, mAP50-95 = 0.582



## Install MMRotate

Run this in a fresh Colab runtime. MMRotate installation can be version
sensitive, so this cell logs versions clearly. If a package resolver fails,
restart runtime and rerun this cell once.

If Colab is using Python 3.12, old OpenMMLab dependency chains may fail with:

`AttributeError: module 'pkgutil' has no attribute 'ImpImporter'`

That is a package compatibility issue, not a dataset issue. The first lines
below upgrade `setuptools`/`pip` before installing `openmim`.



In [ ]:
!pip -q install -U "pip>=24.0" "setuptools>=69.5.1" wheel packaging
!pip -q install -U "requests==2.32.4" "tqdm>=4.67.0" "rich>=13.8.0"
!pip -q install -U openmim
!mim -q install "mmengine>=0.10.0" "mmcv>=2.1.0" "mmdet>=3.2.0"
!pip -q install "git+https://github.com/open-mmlab/mmrotate.git"
!pip -q install pillow pyyaml pandas matplotlib opencv-python



In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

print("Python:", sys.version)
try:
    import torch
    import mmcv
    import mmdet
    import mmrotate
    print("Torch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    print("MMCV:", mmcv.__version__)
    print("MMDetection:", mmdet.__version__)
    print("MMRotate:", mmrotate.__version__)
except Exception as exc:
    print("Version check warning:", exc)




## Paths



In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0"),
    Path("/content/Pol_ISARSpaceTarget1.0"),
    Path("C:/Users/keerthish/Desktop/drdo/Pol_ISARSpaceTarget1.0"),
]
DATASET_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), ROOT_CANDIDATES[0])
print("DATASET_ROOT:", DATASET_ROOT)

LABELTXT_DIR = DATASET_ROOT / "ImageSets" / "labelTxt"
DAT_DIR = DATASET_ROOT / "ImageSets" / "images"
TIFF_DIR = DATASET_ROOT / "TIFFImages" / "AmplitudeImages"

assert LABELTXT_DIR.exists(), f"Missing labels: {LABELTXT_DIR}"
assert DAT_DIR.exists(), f"Missing complex dat files: {DAT_DIR}"

WORK_ROOT = Path("/content/polisar_mmrotate")
DATA_ROOT = WORK_ROOT / "dota_single_vv"
DRIVE_OUT = Path("/content/drive/MyDrive/drdo/polisar_non_yolo_obb_mmrotate")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)




## Prepare DOTA-Style Dataset for MMRotate

MMRotate supports DOTA-style rotated object detection datasets. We convert the
existing Pol-ISAR labels and images into:

```text
dota_single_vv/
  train/images
  train/labelTxt
  val/images
  val/labelTxt
  test/images
  test/labelTxt
```



In [ ]:
def read_dota_label_file(path: Path) -> List[Tuple[List[float], str, int]]:
    rows = []
    if not path.exists() or not path.read_text().strip():
        return rows
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 10:
            continue
        coords = [float(x) for x in parts[:8]]
        cls_name = parts[8]
        difficult = int(float(parts[9]))
        rows.append((coords, cls_name, difficult))
    return rows


class_names = sorted(
    {
        cls
        for split in ["train", "valid", "test"]
        for txt in (LABELTXT_DIR / split).glob("*.txt")
        for _, cls, _ in read_dota_label_file(txt)
    }
)
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("Classes:", class_names)
(DRIVE_OUT / "class_names.json").write_text(json.dumps(class_names, indent=2))


def robust_uint8(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = np.log1p(np.maximum(x, 0.0))
    lo, hi = np.percentile(x, [1, 99])
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (255 * x).astype(np.uint8)


def base_name_candidates(base_name: str) -> List[str]:
    cleaned = re.sub(r"\s+\(\d+\)$", "", base_name)
    return [base_name, cleaned] if cleaned != base_name else [base_name]


def find_first_existing(paths: Sequence[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def read_complex_dat(path: Path, shape: Tuple[int, int] = (256, 256)) -> np.ndarray:
    raw = np.fromfile(path, dtype=np.float32)
    real = raw[0::2].reshape(shape)
    imag = raw[1::2].reshape(shape)
    return real + 1j * imag


def build_single_vv(base_name: str, split: str) -> np.ndarray:
    tiff_path = find_first_existing([TIFF_DIR / f"{candidate}_VV.tiff" for candidate in base_name_candidates(base_name)])
    if tiff_path is not None:
        img = np.array(Image.open(tiff_path).convert("L"))
        return np.stack([img, img, img], axis=-1)
    dat_path = find_first_existing([DAT_DIR / split / f"{candidate}_VV.dat" for candidate in base_name_candidates(base_name)])
    if dat_path is None:
        raise FileNotFoundError(f"No VV image found for {base_name}")
    vv = read_complex_dat(dat_path)
    img = robust_uint8(np.abs(vv))
    return np.stack([img, img, img], axis=-1)


def write_mmrotate_label(src_txt: Path, dst_txt: Path) -> int:
    out_lines = []
    for coords, cls_name, difficult in read_dota_label_file(src_txt):
        coord_text = " ".join(f"{v:.2f}" for v in coords)
        out_lines.append(f"{coord_text} {cls_name} {difficult}")
    dst_txt.parent.mkdir(parents=True, exist_ok=True)
    dst_txt.write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    return len(out_lines)


def build_mmrotate_dota_single_vv(force_rebuild: bool = False) -> Path:
    if DATA_ROOT.exists() and not force_rebuild and any((DATA_ROOT / "train" / "images").glob("*.png")):
        print("Using existing MMRotate dataset:", DATA_ROOT)
        return DATA_ROOT
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)

    split_map = {"train": "train", "valid": "val", "test": "test"}
    summary = []
    for src_split, dst_split in split_map.items():
        img_dir = DATA_ROOT / dst_split / "images"
        ann_dir = DATA_ROOT / dst_split / "labelTxt"
        img_dir.mkdir(parents=True, exist_ok=True)
        ann_dir.mkdir(parents=True, exist_ok=True)
        txt_files = sorted((LABELTXT_DIR / src_split).glob("*.txt"))
        obj_count = 0
        for idx, txt in enumerate(txt_files, start=1):
            base = txt.stem
            img = build_single_vv(base, src_split)
            cv2.imwrite(str(img_dir / f"{base}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            obj_count += write_mmrotate_label(txt, ann_dir / f"{base}.txt")
            if idx % 100 == 0:
                print(dst_split, idx, "/", len(txt_files))
        summary.append({"split": dst_split, "images": len(txt_files), "objects": obj_count})
    summary_df = pd.DataFrame(summary)
    display(summary_df)
    summary_df.to_csv(DRIVE_OUT / "mmrotate_dataset_summary.csv", index=False)
    return DATA_ROOT


DATA_ROOT = build_mmrotate_dota_single_vv(force_rebuild=False)




## Locate MMRotate Install and Configs



In [ ]:
import mmrotate

MMROTATE_DIR = Path(mmrotate.__file__).resolve().parents[1]
print("MMROTATE_DIR:", MMROTATE_DIR)

CONFIG_CANDIDATES = {
    "Rotated_RetinaNet": [
        "configs/rotated_retinanet/rotated_retinanet_obb_r50_fpn_1x_dota_oc.py",
        "configs/rotated_retinanet/rotated_retinanet_obb_r50_fpn_1x_dota_le90.py",
    ],
    "Rotated_Faster_RCNN": [
        "configs/roi_trans/rotated_faster_rcnn_r50_fpn_1x_dota_le90.py",
    ],
    "RoI_Transformer": [
        "configs/roi_trans/roi_trans_r50_fpn_1x_dota_le90.py",
    ],
    "Oriented_RCNN": [
        "configs/oriented_rcnn/oriented_rcnn_r50_fpn_1x_dota_le90.py",
    ],
    "R3Det": [
        "configs/r3det/r3det_r50_fpn_1x_dota_oc.py",
    ],
}


def find_config(rel_paths: Sequence[str]) -> Path | None:
    for rel in rel_paths:
        path = MMROTATE_DIR / rel
        if path.exists():
            return path
    return None


available_methods = []
for method, rels in CONFIG_CANDIDATES.items():
    cfg = find_config(rels)
    print(method, "->", cfg)
    if cfg is not None:
        available_methods.append((method, cfg))

assert available_methods, "No MMRotate configs found. Check MMRotate installation."




## Generate Custom Configs

The generated configs override:

- number of classes
- dataset root and train/val/test paths
- image size `256 x 256`
- shorter epoch schedule for Colab
- output work directory



In [ ]:
EPOCHS = 24
SAMPLES_PER_GPU = 2
WORKERS_PER_GPU = 2
LR = 0.0025

CUSTOM_CFG_DIR = WORK_ROOT / "configs"
CUSTOM_CFG_DIR.mkdir(parents=True, exist_ok=True)


def make_custom_config(method: str, base_cfg: Path) -> Path:
    out_cfg = CUSTOM_CFG_DIR / f"{method}_polisar_single_vv.py"
    class_tuple = "(" + ", ".join(repr(c) for c in class_names) + ",)"
    angle_version = "oc" if "_oc" in base_cfg.name else "le90"
    if method in {"Rotated_RetinaNet", "R3Det"}:
        model_override = f"""
# One-stage rotated detectors use model.bbox_head.
model = dict(
    bbox_head=dict(num_classes={len(class_names)})
)
"""
    elif method == "RoI_Transformer":
        model_override = f"""
# RoI Transformer has two bbox heads in the cascade-style roi_head.
model = dict(
    roi_head=dict(
        bbox_head=[
            dict(num_classes={len(class_names)}),
            dict(num_classes={len(class_names)})
        ]
    )
)
"""
    else:
        model_override = f"""
# Two-stage rotated detectors use model.roi_head.bbox_head.
model = dict(
    roi_head=dict(
        bbox_head=dict(num_classes={len(class_names)})
    )
)
"""
    content = f"""
_base_ = '{base_cfg.as_posix()}'

angle_version = '{angle_version}'
classes = {class_tuple}
data_root = r'{DATA_ROOT.as_posix()}/'

img_norm_cfg = dict(mean=[123.675, 116.28, 103.53], std=[58.395, 57.12, 57.375], to_rgb=True)
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='RResize', img_scale=(256, 256)),
    dict(type='RRandomFlip', flip_ratio=0.5, version=angle_version),
    dict(type='Normalize', **img_norm_cfg),
    dict(type='Pad', size_divisor=32),
    dict(type='DefaultFormatBundle'),
    dict(type='Collect', keys=['img', 'gt_bboxes', 'gt_labels'])
]
test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='MultiScaleFlipAug', img_scale=(256, 256), flip=False, transforms=[
        dict(type='RResize'),
        dict(type='Normalize', **img_norm_cfg),
        dict(type='Pad', size_divisor=32),
        dict(type='DefaultFormatBundle'),
        dict(type='Collect', keys=['img'])
    ])
]

data = dict(
    samples_per_gpu={SAMPLES_PER_GPU},
    workers_per_gpu={WORKERS_PER_GPU},
    train=dict(
        type='DOTADataset',
        ann_file=data_root + 'train/labelTxt/',
        img_prefix=data_root + 'train/images/',
        classes=classes,
        pipeline=train_pipeline,
        version=angle_version),
    val=dict(
        type='DOTADataset',
        ann_file=data_root + 'val/labelTxt/',
        img_prefix=data_root + 'val/images/',
        classes=classes,
        pipeline=test_pipeline,
        version=angle_version),
    test=dict(
        type='DOTADataset',
        ann_file=data_root + 'test/labelTxt/',
        img_prefix=data_root + 'test/images/',
        classes=classes,
        pipeline=test_pipeline,
        version=angle_version)
)

{model_override}
optimizer = dict(lr={LR})
runner = dict(type='EpochBasedRunner', max_epochs={EPOCHS})
evaluation = dict(interval=1, metric='mAP')
checkpoint_config = dict(interval=1)
log_config = dict(interval=50)
work_dir = r'{(DRIVE_OUT / "runs_mmrotate" / method).as_posix()}'
"""
    out_cfg.write_text(content)
    return out_cfg


custom_configs = []
for method, base_cfg in available_methods:
    cfg = make_custom_config(method, base_cfg)
    custom_configs.append((method, cfg))
    print(method, "custom config:", cfg)




## Train and Evaluate Non-YOLO OBB Methods

This cell continues even if one method fails, and saves a live CSV.



In [ ]:
TRAIN_SCRIPT = MMROTATE_DIR / "tools" / "train.py"
TEST_SCRIPT = MMROTATE_DIR / "tools" / "test.py"

assert TRAIN_SCRIPT.exists(), TRAIN_SCRIPT
assert TEST_SCRIPT.exists(), TEST_SCRIPT


def run_command(cmd: List[str], log_path: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        return proc.wait()


results = []
for method, cfg in custom_configs:
    started = time.time()
    work_dir = DRIVE_OUT / "runs_mmrotate" / method
    train_log = work_dir / "train.log"
    test_log = work_dir / "test.log"
    row = {"method": method, "framework": "MMRotate", "input_variant": "single_vv", "status": "started", "work_dir": str(work_dir)}
    try:
        code = run_command([sys.executable, str(TRAIN_SCRIPT), str(cfg), "--work-dir", str(work_dir)], train_log)
        if code != 0:
            raise RuntimeError(f"train.py exited with code {code}")

        ckpts = sorted(work_dir.glob("*.pth"), key=lambda p: p.stat().st_mtime)
        if not ckpts:
            raise FileNotFoundError(f"No checkpoint found in {work_dir}")
        ckpt = ckpts[-1]
        code = run_command([sys.executable, str(TEST_SCRIPT), str(cfg), str(ckpt), "--eval", "mAP"], test_log)
        if code != 0:
            raise RuntimeError(f"test.py exited with code {code}")

        row.update({"status": "ok", "checkpoint": str(ckpt), "elapsed_min": (time.time() - started) / 60})
    except Exception as exc:
        row.update({"status": "failed", "error": repr(exc), "elapsed_min": (time.time() - started) / 60})
        print("FAILED:", method, exc)
    results.append(row)
    pd.DataFrame(results).to_csv(DRIVE_OUT / "non_yolo_obb_method_comparison_live.csv", index=False)

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(DRIVE_OUT / "non_yolo_obb_method_comparison_final.csv", index=False)
(DRIVE_OUT / "non_yolo_obb_method_comparison_final.json").write_text(json.dumps(results, indent=2))




## Parse Logs for mAP Values

MMRotate prints mAP in the test log. This helper extracts the last visible
`mAP` number from each log so the guide can see a compact table.



In [ ]:
def extract_last_map(log_path: Path) -> float | None:
    if not log_path.exists():
        return None
    text = log_path.read_text(errors="ignore")
    matches = re.findall(r"mAP[:= ]+([0-9]*\\.?[0-9]+)", text)
    if matches:
        return float(matches[-1])
    matches = re.findall(r"\\|\\s*mAP\\s*\\|\\s*([0-9]*\\.?[0-9]+)", text)
    if matches:
        return float(matches[-1])
    return None


summary_rows = []
for row in results:
    method = row["method"]
    test_log = DRIVE_OUT / "runs_mmrotate" / method / "test.log"
    summary_rows.append(
        {
            "method": method,
            "framework": "MMRotate",
            "input_variant": "single_vv",
            "status": row["status"],
            "test_mAP_from_log": extract_last_map(test_log),
            "work_dir": row["work_dir"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(DRIVE_OUT / "non_yolo_obb_map_summary.csv", index=False)




## Reference YOLO Results

These are previous completed results, included only for comparison after the
non-YOLO methods complete.



In [ ]:
reference = pd.DataFrame(
    [
        {
            "method": "YOLO11n-OBB",
            "framework": "Ultralytics",
            "input_variant": "single_vv",
            "test_precision": 0.804,
            "test_recall": 0.819,
            "test_mAP50": 0.821,
            "test_mAP50_95": 0.582,
        },
        {
            "method": "YOLO11n-OBB",
            "framework": "Ultralytics",
            "input_variant": "pauli_fusion",
            "test_precision": 0.828,
            "test_recall": 0.739,
            "test_mAP50": 0.783,
            "test_mAP50_95": 0.573,
        },
        {
            "method": "YOLO11n-OBB",
            "framework": "Ultralytics",
            "input_variant": "learned_fusion",
            "test_precision": 0.770,
            "test_recall": 0.683,
            "test_mAP50": 0.743,
            "test_mAP50_95": 0.518,
        },
    ]
)
display(reference)
reference.to_csv(DRIVE_OUT / "yolo_reference_results.csv", index=False)
